# R4 - Implementación del segmentador baseline U-Net

Este notebook implementa y verifica el modelo U-Net utilizado como **segmentador baseline** para imágenes ecográficas 2D de tumores ováricos.

El objetivo de esta primera configuración es obtener una referencia experimental reproducible antes de incorporar técnicas de optimización. Por ello, se mantiene fija la preparación de entrada validada previamente, se utiliza una única partición experimental común para las etapas posteriores y se evita utilizar el conjunto oficial de validación durante el desarrollo del modelo.

## 1. Configuración del entorno

Se localiza automáticamente la raíz del proyecto para evitar rutas absolutas dependientes del equipo local. También se fija una semilla para favorecer la reproducibilidad de la inicialización del modelo, del orden de los lotes y de las operaciones aleatorias empleadas durante el experimento.


In [13]:
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from IPython.display import display
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset


# LOCALIZACIÓN DEL PROYECTO

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = None

for candidate in [CURRENT_DIR, *CURRENT_DIR.parents]:
    expected_dataset = candidate / "data" / "processed" / "MMOTU_OTU_2D"

    if expected_dataset.exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError( "No se pudo localizar la raíz del proyecto que contiene " "data/processed/MMOTU_OTU_2D." )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# MÓDULOS DEL PROYECTO

from src.segmentation.dataset import OvarianTumorSegmentationDataset
from src.segmentation.unet import UNet


# RUTAS

DATASET_DIR = PROJECT_ROOT / "data" / "processed" / "MMOTU_OTU_2D"

TRAIN_IMAGES_DIR = DATASET_DIR / "train" / "images"
TRAIN_MASKS_DIR = DATASET_DIR / "train" / "masks"

FINAL_VAL_IMAGES_DIR = DATASET_DIR / "validation" / "images"
FINAL_VAL_MASKS_DIR = DATASET_DIR / "validation" / "masks"

METADATA_PATH = DATASET_DIR / "metadata.csv"

SPLITS_DIR = PROJECT_ROOT / "configs" / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTAL_SPLIT_PATH = SPLITS_DIR / "mmotu_experimental_split.csv"


# CONFIGURACIÓN BASELINE

IMAGE_SIZE = (256, 256)
BATCH_SIZE = 4

IN_CHANNELS = 3
OUT_CHANNELS = 1

INTERNAL_VAL_RATIO = 0.20
SEED = 42
THRESHOLD = 0.5


# REPRODUCIBILIDAD

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Device:", device)
print("PyTorch:", torch.__version__)

PROJECT_ROOT: C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\proyecto_ia_tumores_ovaricos
Device: cpu
PyTorch: 2.14.0+cpu


## 2. Preparación de los datos y partición experimental

La lectura de imágenes y máscaras se realiza mediante el `Dataset` validado en `01_verificacion_entrada.ipynb`.

La partición experimental es común para las etapas de segmentación y clasificación. Las 1000 imágenes pertenecientes al `train` oficial se dividen de forma estratificada según la clase binaria en:

- `train_internal`: 80 %;
- `val_internal`: 20 %.

Las 469 imágenes del `validation` oficial se registran como `final_validation` y permanecen aisladas durante el desarrollo.

El manifiesto se genera únicamente si todavía no existe. En ejecuciones posteriores, se reutiliza el mismo archivo para impedir que la asignación de muestras cambie entre experimentos.


In [14]:
metadata = pd.read_csv(METADATA_PATH)
included_metadata = metadata.loc[ metadata["status"] == "included" ].copy()

print("Muestras incluidas:", len(included_metadata))

display(
    included_metadata[
        ["split", "binary_class"]
    ]
    .value_counts()
    .rename("n")
    .reset_index()
)

Muestras incluidas: 1469


,split,binary_class,n
0,train,benign,891
1,validation,benign,421
2,train,malignant,109
3,validation,malignant,48


In [15]:
# CREACIÓN ÚNICA O CARGA DEL MANIFIESTO

if EXPERIMENTAL_SPLIT_PATH.exists():
    split_df = pd.read_csv(EXPERIMENTAL_SPLIT_PATH)
    print("Se reutiliza la partición experimental existente.")

else:
    official_train = included_metadata.loc[ included_metadata["split"] == "train" ].copy()

    train_internal_df, val_internal_df = train_test_split(
        official_train,
        test_size=INTERNAL_VAL_RATIO,
        random_state=SEED,
        stratify=official_train["binary_label"],
    )

    final_validation_df = included_metadata.loc[ included_metadata["split"] == "validation" ].copy()
    train_internal_df = train_internal_df.copy()
    val_internal_df = val_internal_df.copy()
    final_validation_df = final_validation_df.copy()

    train_internal_df["experimental_split"] = "train_internal"
    val_internal_df["experimental_split"] = "val_internal"
    final_validation_df["experimental_split"] = "final_validation"

    split_df = pd.concat( [ train_internal_df, val_internal_df, final_validation_df, ], ignore_index=True, )
    split_df = split_df[
        [
            "image_id",
            "image_name",
            "split",
            "original_class",
            "binary_label",
            "binary_class",
            "experimental_split",
        ]
    ].rename( columns={"split": "source_split"} )
    split_df.to_csv( EXPERIMENTAL_SPLIT_PATH, index=False, encoding="utf-8-sig", )
    print("Partición experimental creada y guardada.")

print(EXPERIMENTAL_SPLIT_PATH)


Se reutiliza la partición experimental existente.
C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\proyecto_ia_tumores_ovaricos\configs\splits\mmotu_experimental_split.csv


### 2.1. Verificación del manifiesto experimental

- cada imagen aparezca una única vez;
- existan 800 muestras de entrenamiento interno y 200 muestras de validación interna;
- las 469 muestras oficiales de validación permanezcan asignadas a evaluación final;


In [16]:
required_columns = {
    "image_id",
    "image_name",
    "source_split",
    "binary_label",
    "binary_class",
    "experimental_split",
}

missing_columns = required_columns - set(split_df.columns)

if missing_columns:
    raise ValueError( f"Faltan columnas en el manifiesto: {sorted(missing_columns)}" )
if split_df["image_id"].duplicated().any():
    raise ValueError( "El manifiesto contiene image_id duplicados." )

expected_counts = {
    "train_internal": 800,
    "val_internal": 200,
    "final_validation": 469,
}
observed_counts = (
    split_df["experimental_split"]
    .value_counts()
    .to_dict()
)
assert observed_counts == expected_counts, ( f"Distribución inesperada: {observed_counts}" )

development_rows = split_df[ split_df["experimental_split"].isin( ["train_internal", "val_internal"] ) ]
assert (development_rows["source_split"] == "train").all()

final_rows = split_df[ split_df["experimental_split"] == "final_validation" ]
assert (final_rows["source_split"] == "validation").all()

manifest_ids = set( split_df["image_id"].astype(str) )

metadata_ids = set( included_metadata["image_id"].astype(str) )
assert manifest_ids == metadata_ids

display(
    split_df[
        ["experimental_split", "binary_class"]
    ]
    .value_counts()
    .rename("n")
    .reset_index()
)

print("Manifiesto experimental verificado correctamente.")

,experimental_split,binary_class,n
0,train_internal,benign,713
1,final_validation,benign,421
2,val_internal,benign,178
3,train_internal,malignant,87
4,final_validation,malignant,48
5,val_internal,malignant,22


Manifiesto experimental verificado correctamente.


### 2.2. Construcción de los subconjuntos de desarrollo

El `Dataset` continúa leyendo las 1000 muestras del `train` oficial y el manifiesto determina qué identificadores pertenecen a `train_internal` y cuáles a `val_internal`. De este modo, la separación experimental se controla de forma explícita mediante IDs y puede reutilizarse en todos los experimentos posteriores.


In [17]:
full_train_dataset = OvarianTumorSegmentationDataset(
    images_dir=TRAIN_IMAGES_DIR,
    masks_dir=TRAIN_MASKS_DIR,
    image_size=IMAGE_SIZE,
)

id_to_index = {
    str(image_id): index
    for index, image_id in enumerate(full_train_dataset.ids)
}
train_internal_ids = (
    split_df.loc[ split_df["experimental_split"] == "train_internal", "image_id", ]
    .astype(str)
    .tolist()
)
val_internal_ids = (
    split_df.loc[ split_df["experimental_split"] == "val_internal", "image_id", ]
    .astype(str)
    .tolist()
)

development_ids = set( train_internal_ids + val_internal_ids )
missing_dataset_ids = development_ids - set(id_to_index)

if missing_dataset_ids:
    raise ValueError( f"IDs del manifiesto no encontrados en el Dataset: " f"{sorted(missing_dataset_ids)[:10]}" )

overlap = set(train_internal_ids) & set(val_internal_ids)
assert len(overlap) == 0, ( "Existe solapamiento entre train_internal y val_internal." )

train_internal_indices = [
    id_to_index[image_id]
    for image_id in train_internal_ids
]
val_internal_indices = [
    id_to_index[image_id]
    for image_id in val_internal_ids
]

train_internal_dataset = Subset( full_train_dataset, train_internal_indices, )
val_internal_dataset = Subset( full_train_dataset, val_internal_indices, )

print("Train oficial:", len(full_train_dataset))
print("Train interno:", len(train_internal_dataset))
print("Validation interna:", len(val_internal_dataset))
print("Solapamiento:", len(overlap))

Train oficial: 1000
Train interno: 800
Validation interna: 200
Solapamiento: 0


### 2.3. DataLoaders

Los `DataLoader` agrupan las muestras en lotes para alimentar la red. El generador asociado al `DataLoader` de entrenamiento utiliza la misma semilla del experimento, de modo que el orden aleatorio sea reproducible entre ejecuciones equivalentes.


In [18]:
train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_internal_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    generator=train_generator,
)
internal_val_loader = DataLoader(
    val_internal_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

print("Batches train:", len(train_loader))
print("Batches validation interna:", len(internal_val_loader))

batch = next(iter(train_loader))

print("Images:", batch["image"].shape)
print("Masks:", batch["mask"].shape)

Batches train: 200
Batches validation interna: 50
Images: torch.Size([4, 3, 256, 256])
Masks: torch.Size([4, 1, 256, 256])


## 3. Arquitectura U-Net baseline

El segmentador baseline se implementa mediante una arquitectura U-Net de tipo encoder-decoder con conexiones de salto (*skip connections*).

Para mantener una configuración baseline sencilla no se incorporan capas de Batch Normalization ni otras modificaciones orientadas a optimizar el entrenamiento.

In [19]:
model = UNet( in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS, ).to(device)

total_params = sum(
    p.numel()
    for p in model.parameters()
)
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Modelo:", model.__class__.__name__)
print(f"Parámetros totales: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")


Modelo: UNet
Parámetros totales: 31,031,745
Parámetros entrenables: 31,031,745


### 3.1. Verificación estructural de U-Net

Antes de definir el entrenamiento se realiza una propagación hacia adelante sin modificar los pesos.

El objetivo es confirmar que un lote con forma `[B, 3, 256, 256]` produce logits con forma `[B, 1, 256, 256]`.


In [20]:
smoke_batch = next(iter(train_loader))
smoke_images = smoke_batch["image"].to(device)
model.eval()

with torch.no_grad():
    smoke_logits = model(smoke_images)

print("Entrada:", smoke_images.shape)
print("Salida:", smoke_logits.shape)

assert smoke_logits.shape == ( smoke_images.shape[0], OUT_CHANNELS, IMAGE_SIZE[0], IMAGE_SIZE[1], )


Entrada: torch.Size([4, 3, 256, 256])
Salida: torch.Size([4, 1, 256, 256])


## 4. Función de pérdida

Para el baseline, se utiliza `BCEWithLogitsLoss`, adecuada para segmentación binaria. Esta funcion combina internamente la transformación sigmoide y la entropía cruzada binaria, por lo que **no se aplica `sigmoid` a la salida antes de calcular la pérdida**.


In [21]:
criterion = nn.BCEWithLogitsLoss()

print("Función de pérdida:")
print(criterion)

Función de pérdida:
BCEWithLogitsLoss()


## 5. Métricas de evaluación: Dice e IoU

La función de pérdida guía la actualización de los parámetros, mientras que Dice e IoU se utilizan para interpretar la calidad espacial de la segmentación. Para calcular estas métricas, los logits se transforman en probabilidades mediante `sigmoid` y posteriormente se aplica un umbral de 0.5 para generar una máscara binaria.


In [22]:
def dice_iou_scores( logits, targets, threshold=THRESHOLD, eps=1e-7, ):
    """
    Devuelve Dice e IoU por imagen del batch.
    """

    probabilities = torch.sigmoid(logits)
    predictions = ( probabilities >= threshold ).float()
    dimensions = (1, 2, 3)

    intersection = ( predictions * targets ).sum(dim=dimensions)
    pred_area = predictions.sum(dim=dimensions)
    target_area = targets.sum(dim=dimensions)

    dice = ( 2.0 * intersection + eps ) / ( pred_area + target_area + eps )
    union = ( pred_area + target_area - intersection )
    iou = ( intersection + eps ) / ( union + eps )

    return dice, iou

## 6. Verificación de pérdida y métricas antes del entrenamiento

Se procesa un lote sin realizar actualizaciones de pesos para comprobar que la salida de U-Net es compatible con las máscaras y que la pérdida, Dice e IoU pueden calcularse correctamente.

Los valores obtenidos aquí **no representan el desempeño del segmentador**, ya que el modelo todavía no ha sido entrenado.


In [23]:
batch = next(iter(train_loader))
images = batch["image"].to(device)
masks = batch["mask"].to(device)

model.eval()

with torch.no_grad():
    logits = model(images)
    loss = criterion( logits, masks, )
    dice_values, iou_values = dice_iou_scores( logits, masks, )

print("Dimensiones:")
print("Images:", images.shape)
print("Masks:", masks.shape)
print("Logits:", logits.shape)

print("\nResultados antes del entrenamiento:")
print(f"Loss: {loss.item():.6f}")
print(f"Dice promedio: {dice_values.mean().item():.6f}")
print(f"IoU promedio:  {iou_values.mean().item():.6f}")

assert torch.isfinite(loss)
assert 0.0 <= dice_values.mean().item() <= 1.0
assert 0.0 <= iou_values.mean().item() <= 1.0


Dimensiones:
Images: torch.Size([4, 3, 256, 256])
Masks: torch.Size([4, 1, 256, 256])
Logits: torch.Size([4, 1, 256, 256])

Resultados antes del entrenamiento:
Loss: 0.656410
Dice promedio: 0.000000
IoU promedio:  0.000000


In [24]:
with torch.no_grad():

    probabilities = torch.sigmoid(logits)

    predictions = (
        probabilities >= THRESHOLD
    ).float()


print("Probabilidad mínima:",
      probabilities.min().item())

print("Probabilidad máxima:",
      probabilities.max().item())

print("Probabilidad media:",
      probabilities.mean().item())


predicted_positive_pixels = predictions.sum().item()
total_pixels = predictions.numel()

print(
    "\nPíxeles predichos como lesión:",
    int(predicted_positive_pixels)
)

print(
    "Porcentaje de píxeles predichos como lesión:",
    100 * predicted_positive_pixels / total_pixels
)

print(
    "\nPíxeles de lesión reales en el batch:",
    int(masks.sum().item())
)

Probabilidad mínima: 0.4693833887577057
Probabilidad máxima: 0.4721702039241791
Probabilidad media: 0.47085586190223694

Píxeles predichos como lesión: 0
Porcentaje de píxeles predichos como lesión: 0.0

Píxeles de lesión reales en el batch: 44774


## 7. Configuración del optimizador

Para el baseline, se utiliza el optimizador Adam con una tasa de aprendizaje fija de `1e-4`.

No se incorporan schedulers, weight decay ni procedimientos de búsqueda de hiperparámetros en esta primera configuración. El objetivo es mantener un experimento base reproducible antes de evaluar posibles estrategias de optimización.

In [25]:
LEARNING_RATE = 1e-4
optimizer = torch.optim.Adam( model.parameters(), lr=LEARNING_RATE, )

print("Optimizador:", optimizer.__class__.__name__)
print("Learning rate:", LEARNING_RATE)

Optimizador: Adam
Learning rate: 0.0001


## 8. Verificación de la retropropagación

Antes de ejecutar una época completa, se realiza una actualización utilizando un único lote de entrenamiento. Esta actualización tiene únicamente una finalidad técnica y no formará parte del entrenamiento experimental definitivo.

In [26]:
test_batch = next(iter(train_loader))
test_images = test_batch["image"].to(device)
test_masks = test_batch["mask"].to(device)

model.train()

# Eliminar gradientes acumulados anteriormente
optimizer.zero_grad()

# Forward
test_logits = model(test_images)

# Loss
test_loss = criterion( test_logits, test_masks, )

# Backpropagation
test_loss.backward()

# Actualización de pesos
optimizer.step()

print( f"Backpropagation completada correctamente. " f"Loss: {test_loss.item():.6f}" )

Backpropagation completada correctamente. Loss: 0.649475


### Reinicialización del modelo

La prueba de retropropagación anterior produjo una actualización de los parámetros del modelo. Como dicha ejecución tuvo únicamente una finalidad de verificación, la arquitectura y el optimizador se reinicializan antes de iniciar el entrenamiento baseline.

In [27]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model = UNet( in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS, ).to(device)
optimizer = torch.optim.Adam( model.parameters(), lr=LEARNING_RATE, )

print("Modelo reinicializado.")
print("Optimizador reinicializado.")

Modelo reinicializado.
Optimizador reinicializado.


## 10. Entrenamiento de una época

In [28]:
def train_one_epoch( model, loader, criterion, optimizer, device, ):
    model.train()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    total_samples = 0

    for batch in loader:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)

        batch_size = images.size(0)

        # Reiniciar gradientes
        optimizer.zero_grad()

        # Forward
        logits = model(images)

        # Loss
        loss = criterion( logits, masks, )

        # Backpropagation
        loss.backward()

        # Actualización de parámetros
        optimizer.step()

        # Métricas
        with torch.no_grad():
            dice_values, iou_values = dice_iou_scores( logits, masks, )

        total_loss += ( loss.item() * batch_size )
        total_dice += ( dice_values.sum().item() )
        total_iou += ( iou_values.sum().item() )
        total_samples += batch_size

    return {"loss": total_loss / total_samples, 
            "dice": total_dice / total_samples, 
            "iou": total_iou / total_samples, }

## 11. Validación interna

Al finalizar cada época de entrenamiento, el modelo se evalúa sobre las 200 muestras de `val_internal`. Durante esta etapa se deshabilita el cálculo de gradientes y no se ejecuta ninguna actualización de los parámetros.

In [29]:
def validate_one_epoch( model, loader, criterion, device, ):
    model.eval()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    total_samples = 0

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device)
            masks = batch["mask"].to(device)

            batch_size = images.size(0)
            logits = model(images)
            loss = criterion( logits, masks, )
            dice_values, iou_values = dice_iou_scores( logits, masks, )

            total_loss += ( loss.item() * batch_size )
            total_dice += ( dice_values.sum().item() )
            total_iou += ( iou_values.sum().item() )
            total_samples += batch_size


    return {
        "loss": total_loss / total_samples,
        "dice": total_dice / total_samples,
        "iou": total_iou / total_samples,
    }

## 12. Ejecución de una época de prueba

Esta prueba permite verificar el funcionamiento conjunto del ciclo de entrenamiento y medir el tiempo de ejecución requerido por una época completa.

In [30]:
import time

start_time = time.time()

train_metrics = train_one_epoch(
    model=model,
    loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
)

val_metrics = validate_one_epoch(
    model=model,
    loader=internal_val_loader,
    criterion=criterion,
    device=device,
)


elapsed_time = time.time() - start_time


print("ÉPOCA DE PRUEBA")
print("=" * 45)

print("\nTRAIN")
print(f"Loss : {train_metrics['loss']:.6f}")
print(f"Dice : {train_metrics['dice']:.6f}")
print(f"IoU  : {train_metrics['iou']:.6f}")

print("\nVALIDATION INTERNA")
print(f"Loss : {val_metrics['loss']:.6f}")
print(f"Dice : {val_metrics['dice']:.6f}")
print(f"IoU  : {val_metrics['iou']:.6f}")

print("\nTIEMPO")
print(
    f"{elapsed_time / 60:.2f} minutos"
)

KeyboardInterrupt: 